# Soluções dos Exercícios — Trabalhando com Várias Fontes de Dados

> Tente resolver os exercícios por conta própria antes de consultar estas soluções!


## Setup

Execute esta célula antes de começar.


In [1]:
import sys
!{sys.executable} -m pip install openpyxl pdfplumber PyPDF2 reportlab python-docx -q

import pandas as pd
import numpy as np
import json
import csv
import sqlite3
import requests
import xml.etree.ElementTree as ET

print("Pronto!")



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Pronto!


### Exercício 1: CSV — Cálculo de Estoque

Crie um CSV com produtos, calcule valor total do estoque (preço × quantidade) e salve em novo CSV.


In [2]:
# Solução
import pandas as pd

# Criando DataFrame com produtos
estoque = pd.DataFrame({
    'produto': ['Notebook', 'Mouse', 'Teclado', 'Monitor', 'Webcam'],
    'preco': [3500.0, 80.0, 150.0, 900.0, 200.0],
    'quantidade': [10, 50, 30, 15, 25]
})

# Salvando CSV original
estoque.to_csv('produtos.csv', index=False)

# Calculando valor total do estoque
estoque['valor_total'] = estoque['preco'] * estoque['quantidade']

# Salvando resultado
estoque.to_csv('estoque_total.csv', index=False)

print("Estoque calculado:")
display(estoque)
print("\nValor total do estoque: R$ {:.2f}".format(estoque['valor_total'].sum()))


Estoque calculado:


,produto,preco,quantidade,valor_total
0,Notebook,3500.0,10,35000.0
1,Mouse,80.0,50,4000.0
2,Teclado,150.0,30,4500.0
3,Monitor,900.0,15,13500.0
4,Webcam,200.0,25,5000.0



Valor total do estoque: R$ 62000.00


### Exercício 2: JSON + API

Requisitar posts da API JSONPlaceholder, converter para DataFrame e salvar userId e title em Excel.


In [3]:
# Solução
import requests
import pandas as pd

# Requisição
resposta = requests.get('https://jsonplaceholder.typicode.com/posts')

if resposta.status_code == 200:
    dados = resposta.json()
    df_posts = pd.DataFrame(dados)[['userId', 'title']]

    # Salvando em Excel
    df_posts.to_excel('posts_api.xlsx', index=False)
    print(f"{len(df_posts)} registros salvos em posts_api.xlsx")
    display(df_posts.head())


100 registros salvos em posts_api.xlsx


,userId,title
0,1,sunt aut facere repellat provident occaecati e...
1,1,qui est esse
2,1,ea molestias quasi exercitationem repellat qui...
3,1,eum et est occaecati
4,1,nesciunt quas odio


### Exercício 3: SQLite — JOIN entre tabelas

Criar tabelas `clientes` e `pedidos`, inserir dados e fazer JOIN.


In [ ]:
# Solução
import sqlite3
import pandas as pd

conexao = sqlite3.connect('vendas.db')
cursor = conexao.cursor()

# Criando tabelas
cursor.execute('''
CREATE TABLE IF NOT EXISTS clientes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cidade TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS pedidos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    cliente_id INTEGER,
    valor REAL,
    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
)
''')

# Inserindo clientes
clientes = [
    ('Ana Silva', 'Sao Paulo'),
    ('Bruno Costa', 'Rio de Janeiro'),
    ('Carla Souza', 'Belo Horizonte')
]
cursor.executemany('INSERT INTO clientes (nome, cidade) VALUES (?, ?)', clientes)

# Inserindo pedidos
pedidos = [
    (1, 1500.0),
    (1, 800.0),
    (2, 3200.0),
    (3, 450.0),
    (3, 2100.0)
]
cursor.executemany('INSERT INTO pedidos (cliente_id, valor) VALUES (?, ?)', pedidos)
conexao.commit()

# JOIN: nome do cliente com seus pedidos
query = '''
SELECT c.nome, c.cidade, p.id AS pedido_id, p.valor
FROM clientes c
JOIN pedidos p ON c.id = p.cliente_id
ORDER BY c.nome
'''
df_join = pd.read_sql_query(query, conexao)
display(df_join)

conexao.close()


### Exercício 4: XML

Criar XML com lista de livros, ler e exportar para DataFrame.


In [ ]:
# Solução
import xml.etree.ElementTree as ET
import pandas as pd

# Criando XML
root = ET.Element('biblioteca')

livros = [
    {'titulo': '1984', 'autor': 'George Orwell', 'ano': '1949'},
    {'titulo': 'Dom Casmurro', 'autor': 'Machado de Assis', 'ano': '1899'},
    {'titulo': 'O Pequeno Principe', 'autor': 'Antoine de Saint-Exupery', 'ano': '1943'}
]

for livro in livros:
    el = ET.SubElement(root, 'livro')
    ET.SubElement(el, 'titulo').text = livro['titulo']
    ET.SubElement(el, 'autor').text = livro['autor']
    ET.SubElement(el, 'ano').text = livro['ano']

tree = ET.ElementTree(root)
tree.write('livros.xml', encoding='utf-8', xml_declaration=True)

# Lendo XML e exportando para DataFrame
tree = ET.parse('livros.xml')
root = tree.getroot()

dados = []
for livro in root.findall('livro'):
    dados.append({
        'titulo': livro.find('titulo').text,
        'autor': livro.find('autor').text,
        'ano': int(livro.find('ano').text)
    })

df_livros = pd.DataFrame(dados)
display(df_livros)


### Exercício 5: PDF com Tabela

Criar DataFrame, gerar PDF com tabela, extrair dados de volta e comparar.


In [ ]:
# Solução
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import pdfplumber

# DataFrame original
df_vendas = pd.DataFrame({
    'produto': ['Notebook', 'Mouse', 'Teclado', 'Monitor'],
    'quantidade': [10, 50, 30, 15],
    'preco': [3500.0, 80.0, 150.0, 900.0]
})
df_vendas['total'] = df_vendas['quantidade'] * df_vendas['preco']

# Gerando PDF com tabela
doc = SimpleDocTemplate('exercicio_vendas.pdf', pagesize=letter)
story = []
styles = getSampleStyleSheet()
story.append(Paragraph('Relatorio de Vendas', styles['Heading1']))

cabecalho = list(df_vendas.columns)
dados_tabela = [cabecalho] + df_vendas.values.tolist()
dados_tabela_str = [[str(c) for c in linha] for linha in dados_tabela]

tabela = Table(dados_tabela_str)
tabela.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('GRID', (0, 0), (-1, -1), 1, colors.black),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
]))
story.append(tabela)
doc.build(story)

# Extraindo dados do PDF
with pdfplumber.open('exercicio_vendas.pdf') as pdf:
    tabelas_pdf = pdf.pages[0].extract_tables()
    if tabelas_pdf:
        df_extraido = pd.DataFrame(tabelas_pdf[0][1:], columns=tabelas_pdf[0][0])
        # Convertendo colunas numéricas
        for col in ['quantidade', 'preco', 'total']:
            df_extraido[col] = pd.to_numeric(df_extraido[col], errors='coerce')

print("DataFrame original:")
display(df_vendas)
print("\nDataFrame extraido do PDF:")
display(df_extraido)


### Exercício 6: DOCX

Criar documento Word com título, parágrafos formatados e tabela com dados de um DataFrame.


In [ ]:
# Solução
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import pandas as pd

# DataFrame de exemplo
df = pd.DataFrame({
    'Produto': ['Notebook', 'Mouse', 'Teclado'],
    'Vendas': [150, 320, 210],
    'Faturamento': [525000, 25600, 31500]
})

# Criando documento
doc = Document()

titulo = doc.add_heading('Relatorio de Vendas Mensal', 0)
titulo.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph('Este relatorio apresenta os dados de vendas do mes de junho de 2026.')
doc.add_paragraph('Os valores foram consolidados a partir do sistema interno.')

# Parágrafo com destaque
p = doc.add_paragraph()
p.add_run('Destaque: ').bold = True
p.add_run('O produto mais vendido foi ').font.size = Pt(12)
p.add_run('Mouse').bold = True
p.add_run(' com 320 unidades.')

# Tabela
tabela = doc.add_table(rows=len(df) + 1, cols=len(df.columns))
tabela.style = 'Light Grid Accent 1'

for j, col in enumerate(df.columns):
    tabela.rows[0].cells[j].text = col

for i in range(len(df)):
    for j in range(len(df.columns)):
        tabela.rows[i + 1].cells[j].text = str(df.iloc[i, j])

doc.save('exercicio_relatorio.docx')
print('DOCX criado com sucesso!')


### Exercício 7: Upload + Google Drive (Colab)

> Esta solução funciona apenas no Google Colab.

Upload de CSV, remover nulos com `df.dropna()`, salvar no Drive.


In [ ]:
# Solução (Google Colab)
from google.colab import files, drive

# Upload
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

# Ler e limpar
df = pd.read_csv(nome_arquivo)
df_limpo = df.dropna()

# Montar Drive
drive.mount('/content/drive')

# Salvar no Drive
caminho = f'/content/drive/My Drive/{nome_arquivo}'
df_limpo.to_csv(caminho, index=False)

print(f'Dados limpos salvos em: {caminho}')
print(f'Linhas originais: {len(df)} | Apos limpeza: {len(df_limpo)}')


### Exercício 8: Desafio Extra

Combinar API → SQLite → pandas → DOCX → PDF → Excel.


In [ ]:
# Solução
import requests, sqlite3, pandas as pd
from docx import Document
from docx.shared import Pt
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# 1. Buscar dados da API
print("1. Buscando dados da API...")
resposta = requests.get('https://jsonplaceholder.typicode.com/users')
users = resposta.json()
df_users = pd.json_normalize(users)

# 2. Salvar em SQLite
print("2. Salvando em SQLite...")
conexao = sqlite3.connect('desafio.db')
df_users.to_sql('usuarios', conexao, if_exists='replace', index=False)

# 3. Ler do SQLite com pandas
print("3. Lendo do SQLite...")
df = pd.read_sql_query('SELECT id, name, email, address_city FROM usuarios', conexao)
display(df)
conexao.close()

# 4. Criar DOCX com tabela
print("4. Criando DOCX...")
doc = Document()
doc.add_heading('Usuarios - API JSONPlaceholder', 0)

tabela = doc.add_table(rows=len(df) + 1, cols=len(df.columns))
tabela.style = 'Light Grid Accent 1'
for j, col in enumerate(df.columns):
    tabela.rows[0].cells[j].text = col
for i in range(len(df)):
    for j in range(len(df.columns)):
        tabela.rows[i + 1].cells[j].text = str(df.iloc[i, j])
doc.save('desafio_usuarios.docx')

# 5. Criar PDF resumido
print("5. Criando PDF...")
doc_pdf = SimpleDocTemplate('desafio_resumo.pdf', pagesize=letter)
story = []
styles = getSampleStyleSheet()
story.append(Paragraph('Resumo de Usuarios', styles['Heading1']))
story.append(Paragraph(f'Total de usuarios: {len(df)}', styles['Normal']))
cabecalho = list(df.columns)
dados_pdf = [cabecalho] + df.values.tolist()
dados_str = [[str(c) for c in row] for row in dados_pdf]
tabela_pdf = Table(dados_str)
tabela_pdf.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('GRID', (0, 0), (-1, -1), 1, colors.black),
]))
story.append(tabela_pdf)
doc_pdf.build(story)

# 6. Salvar Excel resumido
print("6. Criando Excel...")
with pd.ExcelWriter('desafio_completo.xlsx') as writer:
    df.to_excel(writer, sheet_name='Usuarios', index=False)
    pd.DataFrame({'total_usuarios': [len(df)]}).to_excel(
        writer, sheet_name='Resumo', index=False
    )

print("\nDesafio completo! Arquivos gerados:")
print("  - desafio.db")
print("  - desafio_usuarios.docx")
print("  - desafio_resumo.pdf")
print("  - desafio_completo.xlsx")
